# 16 - Generate prediction - gemma2 (GGUF)

### Setup

In [1]:
import subprocess, time, os, requests

In [2]:
subprocess.run("apt-get install -y zstd", shell=True, check=True)
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 124 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (4,988 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 125186 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [3]:
import pandas as pd
from datasets import load_dataset
from huggingface_hub import HfApi

## 1. Konfigurasi

In [4]:
def get_hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_Faiss")
    except ImportError:
        token = os.environ.get("HF_Faiss")
        if token is None:
            raise RuntimeError("HF_TOKEN tidak ditemukan.")
        return token
 
HF_TOKEN = get_hf_token()
HF_REPO = "Makaareeem/publikasi-rag-finetuning-dataset"
OUTPUT_DIR = os.environ.get("OUTPUT_DIR", "/kaggle/working" if os.path.isdir("/kaggle/working") else "/workspace")
OLLAMA_URL = "http://localhost:11434"
 
MODEL_KEY = "gemma2"
GGUF_REPO_BASE = "Makaareeem/gemma2-base-gguf"
GGUF_REPO_FT = "Makaareeem/gemma2-finetuned-gguf"
SYSTEM_ROLE = False
MAX_NEW_TOKENS = 512
REPEAT_PENALTY = 1.0 

SYSTEM_PROMPT = (
    "Anda adalah asisten yang menjawab pertanyaan seputar publikasi statistik BPS "
    "berdasarkan konteks yang diberikan. Kutip bagian konteks yang relevan sebelum menjawab."
)

## 2. Modelfile

In [5]:
def write_modelfile(gguf_repo, path):
    content = f"""FROM hf.co/{gguf_repo}
PARAMETER temperature 0
PARAMETER num_predict {MAX_NEW_TOKENS}
PARAMETER repeat_penalty {REPEAT_PENALTY}
"""
    with open(path, "w") as f:
        f.write(content)
 
modelfile_base_path = f"{OUTPUT_DIR}/Modelfile.gemma2.base"
modelfile_ft_path = f"{OUTPUT_DIR}/Modelfile.gemma2.finetuned"
write_modelfile(GGUF_REPO_BASE, modelfile_base_path)
write_modelfile(GGUF_REPO_FT, modelfile_ft_path)

## 3. Registrasi model di Ollama

In [6]:
subprocess.run(["ollama", "create", "gemma2-base", "-f", modelfile_base_path], check=True)
subprocess.run(["ollama", "create", "gemma2-finetuned", "-f", modelfile_ft_path], check=True)

gathering model components 
pulling manifest ⠋ gathering model components 
pulling manifest ⠙ 
pulling 603f6118b2b4   0% ▕                   ▏    0 B/1.7 GB                  gathering model components 
pulling manifest ⠹ 
pulling 603f6118b2b4   0% ▕                   ▏    0 B/1.7 GB                  gathering model components 
pulling manifest ⠸ 
pulling 603f6118b2b4   0% ▕                   ▏    0 B/1.7 GB                  gathering model components 
pulling manifest ⠼ 
pulling 603f6118b2b4   0% ▕                   ▏  71 KB/1.7 GB                  gathering model components 
pulling manifest ⠴ 
pulling 603f6118b2b4   0% ▕                   ▏  71 KB/1.7 GB                  gathering model components 
pulling manifest ⠦ 
pulling 603f6118b2b4   1% ▕                   ▏  18 MB/1.7 GB                  gathering model components 
pulling manifest ⠧ 
pulling 603f6118b2b4   4% ▕                   ▏  66 MB/1.7 GB                  gathering model components 
pulling manifest ⠇ 
pulling 603f6118

CompletedProcess(args=['ollama', 'create', 'gemma2-finetuned', '-f', '/kaggle/working/Modelfile.gemma2.finetuned'], returncode=0)

## 4. Load dataset

In [7]:
test_ds = load_dataset("csv", data_files=f"hf://datasets/{HF_REPO}/test.csv", split="train", token=HF_TOKEN)
examples = [test_ds[i] for i in range(len(test_ds))]
print(f"test set: {len(examples)} baris")

test.csv: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

test set: 1791 baris


## 5. Format chat

In [8]:
def build_user_content(example):
    return f"Konteks:\n{example['context_joined']}\n\nPertanyaan: {example['instruction']}"
 
def build_ground_truth(example):
    return f"{example['cot']}\n\n{example['response']}"
 
def build_ollama_payload(example, model_name, system_role):
    user_content = build_user_content(example)
    if system_role:
        messages = [{"role": "user", "content": user_content}]
    else:
        messages = [{"role": "user", "content": f"{SYSTEM_PROMPT}\n\n{user_content}"}]
    return {
        "model": model_name,
        "messages": messages,
        "stream": False,
        "options": {"temperature": 0, "num_predict": MAX_NEW_TOKENS, "repeat_penalty": REPEAT_PENALTY},
    }

## 6. Generate via Ollama

In [9]:
def generate_all(model_name, examples, system_role):
    predictions = []
    for i, ex in enumerate(examples):
        payload = build_ollama_payload(ex, model_name, system_role)
        resp = requests.post(f"{OLLAMA_URL}/api/chat", json=payload)
        resp.raise_for_status()
        text = resp.json()["message"]["content"]
        predictions.append(text)
        print(f"generate {i + 1}/{len(examples)}")
    return predictions

### Base model

In [10]:
base_predictions = generate_all("gemma2-base", examples, SYSTEM_ROLE)
base_predictions

generate 1/1791
generate 2/1791
generate 3/1791
generate 4/1791
generate 5/1791
generate 6/1791
generate 7/1791
generate 8/1791
generate 9/1791
generate 10/1791
generate 11/1791
generate 12/1791
generate 13/1791
generate 14/1791
generate 15/1791
generate 16/1791
generate 17/1791
generate 18/1791
generate 19/1791
generate 20/1791
generate 21/1791
generate 22/1791
generate 23/1791
generate 24/1791
generate 25/1791
generate 26/1791
generate 27/1791
generate 28/1791
generate 29/1791
generate 30/1791
generate 31/1791
generate 32/1791
generate 33/1791
generate 34/1791
generate 35/1791
generate 36/1791
generate 37/1791
generate 38/1791
generate 39/1791
generate 40/1791
generate 41/1791
generate 42/1791
generate 43/1791
generate 44/1791
generate 45/1791
generate 46/1791
generate 47/1791
generate 48/1791
generate 49/1791
generate 50/1791
generate 51/1791
generate 52/1791
generate 53/1791
generate 54/1791
generate 55/1791
generate 56/1791
generate 57/1791
generate 58/1791
generate 59/1791
genera

['Berdasarkan konteks yang Anda berikan, saya tidak dapat memberikan rekomendasi dari KPK yang disusun berdasarkan hasil SPI Pendidikan. \n\nKonteks yang Anda berikan hanya menyebutkan bahwa diperlukan pengembangan dan evaluasi berkala program pembiasaan karakter.  **Tidak ada informasi mengenai rekomendasi dari KPK yang disusun berdasarkan hasil SPI Pendidikan.** \n\nUntuk mendapatkan informasi mengenai rekomendasi dari KPK, Anda perlu mencari informasi dari sumber resmi seperti:\n\n* **Website resmi KPK:** https://www.kpk.go.id/\n* **Laporan resmi KPK:** https://www.kpk.go.id/laporan \n* **Berita resmi dari KPK:** https://www.kpk.go.id/berita \n\n\nSemoga membantu! \n',
 'GONE theory yang diprakarsai oleh Jack Bologne. \n\nEmpat faktor pendorong korupsi menurut teori GONE adalah:\n\n* **Greed (keserakahan)**\n* **Opportunity (kesempatan)**\n* **Need (kebutuhan)**\n* **Existence (keberadaan/pengungkapan)** \n',
 'Dalam pendekatan lingkungan maupun kultural masyarakat, solusi yang dapa

### Finetuned model

In [11]:
ft_predictions = generate_all("gemma2-finetuned", examples, SYSTEM_ROLE)
ft_predictions

generate 1/1791
generate 2/1791
generate 3/1791
generate 4/1791
generate 5/1791
generate 6/1791
generate 7/1791
generate 8/1791
generate 9/1791
generate 10/1791
generate 11/1791
generate 12/1791
generate 13/1791
generate 14/1791
generate 15/1791
generate 16/1791
generate 17/1791
generate 18/1791
generate 19/1791
generate 20/1791
generate 21/1791
generate 22/1791
generate 23/1791
generate 24/1791
generate 25/1791
generate 26/1791
generate 27/1791
generate 28/1791
generate 29/1791
generate 30/1791
generate 31/1791
generate 32/1791
generate 33/1791
generate 34/1791
generate 35/1791
generate 36/1791
generate 37/1791
generate 38/1791
generate 39/1791
generate 40/1791
generate 41/1791
generate 42/1791
generate 43/1791
generate 44/1791
generate 45/1791
generate 46/1791
generate 47/1791
generate 48/1791
generate 49/1791
generate 50/1791
generate 51/1791
generate 52/1791
generate 53/1791
generate 54/1791
generate 55/1791
generate 56/1791
generate 57/1791
generate 58/1791
generate 59/1791
genera

['Teks menyatakan bahwa rekomendasi pertama adalah "perlu dikembangkan dan dievaluasi berkala program pembiasaan karakter". Karena itu, rekomendasi pertama KPK adalah pengembangan dan evaluasi berkala program pembiasaan karakter.\n\nRekomendasi pertama yang disusun KPK adalah pengembangan dan evaluasi berkala program pembiasaan karakter.',
 'Teks menyatakan GONE theory diprakarsai oleh Jack Bologne dan menjelaskan empat faktor pendorong korupsi, yaitu Greed, Opportunity, Need, dan Existence. Karena itu, Jack Bologne adalah pencetus teori tersebut, dan keempat faktor tersebut adalah komponen penyusunnya.\n\nJack Bologne adalah pencetus GONE theory, yang menjelaskan bahwa terdapat empat faktor pendorong korupsi, yaitu Greed (keserakahan), Opportunity (kesempatan), Need (kebutuhan), dan Existence (keberadaan/pengungkapan).',
 'Teks menyatakan bahwa solusi dalam pendekatan lingkungan maupun kultural masyarakat tidak bisa lepas dari partisipasi masyarakat. Karena itu, partisipasi masyarakat

## 7. Simpan dan upload

In [12]:
results_df = pd.DataFrame({
    "record_id": [ex.get("record_id", i) for i, ex in enumerate(examples)],
    "instruction": [ex["instruction"] for ex in examples],
    "ans_ref": [build_ground_truth(ex) for ex in examples],
    "ans_pred_base": base_predictions,
    "ans_pred_finetuned": ft_predictions,
})
 
output_path = f"{OUTPUT_DIR}/eval_{MODEL_KEY}_predictions_gguf.csv"
results_df.to_csv(output_path, index=False)
print(f"Tersimpan: {output_path}")
 
api = HfApi(token=HF_TOKEN)
api.upload_file(
    path_or_fileobj=output_path,
    path_in_repo=f"eval_{MODEL_KEY}_predictions_gguf.csv",
    repo_id=HF_REPO,
    repo_type="dataset",
)
 
ollama_proc.terminate()

Tersimpan: /kaggle/working/eval_gemma2_predictions_gguf.csv
